# Held-out Test comparator completion

Đây là phần tiếp nối của **cùng một lần mở Held-out Test** đã hoàn tất
cho ba checkpoint Dense Mapped Logit-KD. Notebook:

- không chạy lại ba checkpoint KD;
- không thay đổi phương pháp, checkpoint, decoding hay threshold;
- chỉ bổ sung Teacher ExpansionNetV2 và ba Student baseline đã khóa;
- ghép báo cáo Teacher–Baseline–KD đúng theo đề cương;
- giữ nguyên `test_open_count = 1`.

In [1]:
import csv
import gc
import hashlib
import json
import math
import pickle
import platform
import statistics
import subprocess
import sys
import time
import uuid
from argparse import Namespace
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
assert INPUT_ROOT.is_dir()

RUN_NAME = (
    "XLKD05_Dense_Mapped_LogitKD_"
    "held_out_test_comparator_completion"
)
OUT = WORK_ROOT / RUN_NAME
AUDITS = OUT / "audits"
RESULTS = OUT / "results"
PREDICTIONS = OUT / "predictions"
MANIFESTS = OUT / "manifests"
for directory in (OUT, AUDITS, RESULTS, PREDICTIONS, MANIFESTS):
    directory.mkdir(parents=True, exist_ok=True)

COMPLETE_MARKER = (
    OUT
    / "XLKD05_Dense_Mapped_LogitKD_"
    "HELD_OUT_TEST_COMPARATORS_COMPLETE.json"
)
assert not COMPLETE_MARKER.exists(), (
    "Comparator completion đã hoàn tất trong working directory này."
)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
    temporary.replace(path)

def candidates(name):
    return sorted(
        (path for path in INPUT_ROOT.rglob(name) if path.is_file()),
        key=lambda path: str(path),
    )

def one_name(name):
    matches = candidates(name)
    assert len(matches) == 1, (
        f"Cần đúng một file {name!r}, tìm thấy {len(matches)}."
    )
    return matches[0]

def one_hash(name, expected):
    matches = [path for path in candidates(name) if sha256(path) == expected]
    assert matches, f"Thiếu {name} có SHA-256 {expected}."
    if len(matches) > 1:
        print(f"INFO: {name} có {len(matches)} bản sao đồng nhất.")
    return matches[0]

# Verify the already-completed KD Test run. This is the only opening.
PRIOR_MARKER_PATH = one_name(
    "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPLETE.json"
)
PRIOR_ROOT = PRIOR_MARKER_PATH.parent
PRIOR_MARKER = read_json(PRIOR_MARKER_PATH)
assert PRIOR_MARKER["status"] == "complete"
assert PRIOR_MARKER["seeds"] == [42, 123, 2026]
assert PRIOR_MARKER["test_images"] == 5000
assert PRIOR_MARKER["test_references"] == 25010
assert PRIOR_MARKER["test_open_count"] == 1
assert PRIOR_MARKER["test_set_used"] is True
assert PRIOR_MARKER["checkpoint_or_seed_selection_on_test"] is False
assert PRIOR_MARKER["threshold_tuning_on_test"] is False
assert PRIOR_MARKER["posthoc_method_changes"] is False

def verify_prior_reference(reference):
    path = PRIOR_ROOT / Path(reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    return path

PRIOR_SUMMARY_PATH = verify_prior_reference(PRIOR_MARKER["summary"])
PRIOR_METRICS_CSV = verify_prior_reference(PRIOR_MARKER["metrics_csv"])
PRIOR_REPORT = verify_prior_reference(PRIOR_MARKER["report"])
PRIOR_MANIFEST_PATH = verify_prior_reference(PRIOR_MARKER["manifest"])
PRIOR_SUMMARY = read_json(PRIOR_SUMMARY_PATH)
assert PRIOR_SUMMARY["test_open_count"] == 1
assert PRIOR_SUMMARY["test_set_used"] is True

with PRIOR_MANIFEST_PATH.open("r", encoding="utf-8", newline="") as handle:
    prior_manifest_rows = list(csv.DictReader(handle))
assert len(prior_manifest_rows) == int(PRIOR_MARKER["manifest"]["rows"]) == 14
for row in prior_manifest_rows:
    path = PRIOR_ROOT / Path(row["relative_path"])
    assert path.is_file()
    assert path.stat().st_size == int(row["bytes"])
    assert sha256(path) == row["sha256"]

PRIOR_OPENING_PATH = (
    PRIOR_ROOT
    / "XLKD05_Dense_Mapped_LogitKD_TEST_OPENED.json"
)
PRIOR_OPENING = read_json(PRIOR_OPENING_PATH)
assert PRIOR_OPENING["open_count"] == 1
assert PRIOR_OPENING["test_set_used"] is True
OPENING_ID = PRIOR_OPENING["opening_id"]

PRIOR_REFERENCES_PATH = (
    PRIOR_ROOT / "results" / "karpathy_held_out_test_references.json"
)
PRIOR_REFERENCES = read_json(PRIOR_REFERENCES_PATH)
assert len(PRIOR_REFERENCES) == 5000
assert sum(len(value) for value in PRIOR_REFERENCES.values()) == 25010

DATASET_COCO = one_hash(
    "dataset_coco.json",
    "af76e819987c9ced8452fdb4da823524e033f7baa097cf40bab83bdab4c469ea",
)
VOCAB_PATH = one_hash(
    "word_vocabulary_minfreq5.json",
    "1c6c7e6146d2700932c82be6f076d93153c757c10a1844076abaea9c11496f97",
)
DECODER_CONTRACT_PATH = one_hash(
    "shared_transformer_decoder_contract.json",
    "46655293c4a64506ea842dd0389715fbf64d19ae389c3ba79d2997fe5c3d62dc",
)

BASELINE_SPECS = {
    42: (
        "s1_mobilenetv3_seed42_best.pt",
        "0a1530beb904725b92d082a0d97febd6da617f9bddd8a4b6ffc817735a33828f",
    ),
    123: (
        "s1_mobilenetv3_seed123_best.pt",
        "7f480b4932c3ee36f55b52949e032a56e861a9afed18d53ef750693e95a70925",
    ),
    2026: (
        "s1_mobilenetv3_seed2026_best.pt",
        "b66413a7c586b653db0189fccbacd0e6336fb049f49baab5434146fe960bc14d",
    ),
}
BASELINE_PATHS = {
    seed: one_hash(name, expected)
    for seed, (name, expected) in BASELINE_SPECS.items()
}

teacher_matches = []
for path in candidates("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = read_json(path)
    except Exception:
        continue
    if payload.get("schema_version") == (
        "expansionnet_v2_offline_bundle_v1"
    ):
        teacher_matches.append((path, payload))
assert len(teacher_matches) == 1
TEACHER_MANIFEST_PATH, TEACHER_MANIFEST = teacher_matches[0]
TEACHER_ROOT = TEACHER_MANIFEST_PATH.parent

def teacher_artifact(record):
    path = (TEACHER_ROOT / record["relative_path"]).resolve()
    assert path.is_relative_to(TEACHER_ROOT.resolve())
    assert path.is_file() and path.stat().st_size == int(record["size_bytes"])
    assert sha256(path) == record["sha256"]
    return path

TEACHER_CHECKPOINT = teacher_artifact(TEACHER_MANIFEST["checkpoint"])
TEACHER_VOCAB = teacher_artifact(TEACHER_MANIFEST["vocabulary"])
for record in TEACHER_MANIFEST["required_files"]:
    teacher_artifact(record)
for record in TEACHER_MANIFEST["integration_patches"]:
    path = TEACHER_ROOT / record["file"]
    assert path.is_file() and sha256(path) == record["sha256"]
TEACHER_SOURCE = TEACHER_ROOT / "ExpansionNet_v2_src"
assert TEACHER_SOURCE.is_dir()

preflight = {
    "schema_version": "held_out_test_comparator_preflight_v1",
    "created_at_utc": now(),
    "status": "passed",
    "continuation_of_opening_id": OPENING_ID,
    "prior_complete_marker": {
        "path": str(PRIOR_MARKER_PATH),
        "sha256": sha256(PRIOR_MARKER_PATH),
    },
    "prior_manifest_rows_verified": len(prior_manifest_rows),
    "baseline_checkpoints": {
        str(seed): {"path": str(path), "sha256": sha256(path)}
        for seed, path in BASELINE_PATHS.items()
    },
    "teacher_checkpoint": {
        "path": str(TEACHER_CHECKPOINT),
        "sha256": sha256(TEACHER_CHECKPOINT),
    },
    "new_test_opening": False,
    "test_open_count": 1,
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
    },
}
write_json(AUDITS / "comparator_cell01_preflight.json", preflight)
print("Prior KD Test             : VERIFIED")
print("Prior manifest            : 14 / 14 VERIFIED")
print("Opening ID                :", OPENING_ID)
print("Baseline checkpoints      : 3 / 3 VERIFIED")
print("Teacher bundle            : VERIFIED")
print("New Test opening          : False")

Prior KD Test             : VERIFIED
Prior manifest            : 14 / 14 VERIFIED
Opening ID                : eaebad11-e151-491d-a1ab-403abd959c00
Baseline checkpoints      : 3 / 3 VERIFIED
Teacher bundle            : VERIFIED
New Test opening          : False


In [2]:
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

assert torch.cuda.is_available(), "Bắt buộc bật Kaggle GPU."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def find_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path)
    matches = sorted(set(matches), key=lambda path: str(path))
    assert len(matches) == 1, (
        f"Cần đúng một thư mục {name}, tìm thấy {len(matches)}."
    )
    return matches[0]

image_leaves = {
    "train2014": find_leaf("train2014", "COCO_train2014_"),
    "val2014": find_leaf("val2014", "COCO_val2014_"),
}
karpathy = read_json(DATASET_COCO)
test_rows = [row for row in karpathy["images"] if row["split"] == "test"]
assert len(test_rows) == 5000

TEST_PATHS = {}
source_references = {}
for row in test_rows:
    image_id = int(row["cocoid"])
    path = image_leaves[row["filepath"]] / row["filename"]
    assert path.is_file()
    TEST_PATHS[image_id] = path
    source_references[str(image_id)] = [
        str(sentence["raw"]).strip() for sentence in row["sentences"]
    ]
assert source_references == PRIOR_REFERENCES
TEST_IMAGE_IDS = tuple(sorted(TEST_PATHS))

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def student_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC, antialias=True
    )
    image = TF.center_crop(image, [224, 224])
    return TF.normalize(TF.to_tensor(image), MEAN, STD)

def teacher_transform(image):
    image = TF.resize(
        image,
        [384, 384],
        interpolation=InterpolationMode.BILINEAR,
        antialias=True,
    )
    return TF.normalize(TF.to_tensor(image), MEAN, STD)

class TestImages(Dataset):
    def __init__(self, transform):
        self.transform = transform

    def __len__(self):
        return len(TEST_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TEST_IMAGE_IDS[index]
        with Image.open(TEST_PATHS[image_id]) as source:
            image = self.transform(source.convert("RGB"))
        return image_id, image

def collate(rows):
    return {
        "image_ids": torch.tensor([row[0] for row in rows], dtype=torch.long),
        "images": torch.stack([row[1] for row in rows]),
    }

STUDENT_LOADER = DataLoader(
    TestImages(student_transform),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate,
)
TEACHER_LOADER = DataLoader(
    TestImages(teacher_transform),
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate,
)

continuation = {
    "schema_version": "held_out_test_comparator_continuation_v1",
    "created_at_utc": now(),
    "continuation_id": str(uuid.uuid4()),
    "continuation_of_opening_id": OPENING_ID,
    "test_images": 5000,
    "test_references": 25010,
    "references_exactly_equal_prior_run": True,
    "new_test_opening": False,
    "test_open_count": 1,
    "test_set_used": True,
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
}
write_json(OUT / "HELD_OUT_TEST_COMPARATOR_CONTINUATION.json", continuation)
write_json(AUDITS / "comparator_cell02_data.json", continuation)
print("Test images               : 5000")
print("References                : 25010 / EXACT PRIOR MATCH")
print("New Test opening          : False")

Test images               : 5000
References                : 25010 / EXACT PRIOR MATCH
New Test opening          : False


In [3]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_large

decoder_config = read_json(DECODER_CONTRACT_PATH)["architecture"]

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        return F.linear(
            self.final_norm(hidden),
            self.token_embedding.weight,
            self.output_bias,
        )

vocabulary = read_json(VOCAB_PATH)
raw = vocabulary["id_to_token"]
ID_TO_TOKEN = (
    {index: token for index, token in enumerate(raw)}
    if isinstance(raw, list)
    else {int(key): value for key, value in raw.items()}
)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
BEAM_SIZE = 3
LENGTH_PENALTY = 0.7
MINIMUM_WORDS = 2
MAXIMUM_STEPS = 31
VOCABULARY_SIZE = 10080

def decode_student(values):
    words = []
    for value in values:
        value = int(value)
        if value == EOS_ID:
            break
        if value not in {PAD_ID, BOS_ID, UNK_ID}:
            words.append(ID_TO_TOKEN[value])
    return " ".join(words)

def beam_search(decoder, memory):
    batch, memory_length, dimension = memory.shape
    assert (memory_length, dimension) == (49, 256)
    tokens = torch.full(
        (batch, BEAM_SIZE, 1), BOS_ID, dtype=torch.long, device=device
    )
    scores = torch.full(
        (batch, BEAM_SIZE), float("-inf"), device=device
    )
    scores[:, 0] = 0.0
    finished = torch.zeros(
        (batch, BEAM_SIZE), dtype=torch.bool, device=device
    )
    lengths = torch.zeros(
        (batch, BEAM_SIZE), dtype=torch.long, device=device
    )
    expanded = (
        memory.unsqueeze(1)
        .expand(-1, BEAM_SIZE, -1, -1)
        .reshape(batch * BEAM_SIZE, memory_length, dimension)
        .contiguous()
    )
    for step in range(MAXIMUM_STEPS):
        flat = tokens.reshape(batch * BEAM_SIZE, -1)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = decoder(flat, expanded, flat.ne(PAD_ID))
        log_probs = F.log_softmax(
            logits[:, -1].float(), dim=-1
        ).reshape(batch, BEAM_SIZE, VOCABULARY_SIZE)
        log_probs[:, :, [PAD_ID, BOS_ID, UNK_ID]] = float("-inf")
        if step < MINIMUM_WORDS:
            log_probs[:, :, EOS_ID] = float("-inf")
        if finished.any():
            log_probs = log_probs.masked_fill(
                finished.unsqueeze(-1), float("-inf")
            )
            indices = finished.nonzero(as_tuple=False)
            log_probs[indices[:, 0], indices[:, 1], EOS_ID] = 0.0
        candidates = scores.unsqueeze(-1) + log_probs
        next_scores, flat_indices = torch.topk(
            candidates.reshape(batch, BEAM_SIZE * VOCABULARY_SIZE),
            k=BEAM_SIZE,
            dim=-1,
        )
        parent = flat_indices // VOCABULARY_SIZE
        next_tokens = flat_indices % VOCABULARY_SIZE
        parent_tokens = torch.gather(
            tokens,
            1,
            parent.unsqueeze(-1).expand(-1, -1, tokens.shape[-1]),
        )
        parent_finished = torch.gather(finished, 1, parent)
        parent_lengths = torch.gather(lengths, 1, parent)
        tokens = torch.cat(
            [parent_tokens, next_tokens.unsqueeze(-1)], dim=-1
        )
        scores = next_scores
        lengths = parent_lengths + (~parent_finished).long()
        finished = parent_finished | next_tokens.eq(EOS_ID)
        if bool(finished.all()):
            break
    normalized = scores / lengths.clamp_min(1).float().pow(LENGTH_PENALTY)
    best = normalized.argmax(dim=1)
    rows = torch.arange(batch, device=device)
    return tokens[rows, best].cpu()

@torch.inference_mode()
def generate_baseline(seed):
    payload = torch.load(
        BASELINE_PATHS[seed], map_location="cpu", weights_only=False
    )
    assert payload["schema_version"] == "phase2_s1_full_training_checkpoint_v1"
    assert int(payload["seed"]) == seed
    assert payload["data_policy"]["test_set_status"] == "sealed"
    assert payload["data_policy"]["test_set_used"] is False
    encoder = S1Encoder()
    decoder = SharedDecoder(decoder_config)
    encoder.load_state_dict(payload["encoder_state_dict"], strict=True)
    decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
    encoder = encoder.to(device).eval()
    decoder = decoder.to(device).eval()
    del payload

    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, batch in enumerate(STUDENT_LOADER, start=1):
        ids = [int(value) for value in batch["image_ids"].tolist()]
        images = batch["images"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
        token_rows = beam_search(decoder, memory)
        for index, image_id in enumerate(ids):
            token_ids = [int(value) for value in token_rows[index].tolist()]
            caption = decode_student(token_ids)
            assert caption and image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "image_id": image_id,
                "caption": caption,
                "token_ids": token_ids,
            }
        if batch_index % 25 == 0 or len(hypotheses) == 5000:
            print(
                f"baseline seed={seed} | {len(hypotheses):04d}/5000 | "
                f"{time.perf_counter() - started:.1f}s",
                flush=True,
            )
    elapsed = time.perf_counter() - started
    del encoder, decoder
    gc.collect()
    torch.cuda.empty_cache()
    return hypotheses, records, elapsed

generated = {}
generation_seconds = {}
for seed in (42, 123, 2026):
    hypotheses, records, elapsed = generate_baseline(seed)
    model_id = f"baseline_seed{seed}"
    generated[model_id] = hypotheses
    generation_seconds[model_id] = elapsed
    write_json(
        PREDICTIONS / f"held_out_test_{model_id}_predictions.json",
        {
            "schema_version": "held_out_test_comparator_predictions_v1",
            "created_at_utc": now(),
            "model_id": model_id,
            "seed": seed,
            "checkpoint_sha256": sha256(BASELINE_PATHS[seed]),
            "split": "held_out_test",
            "images": len(records),
            "beam_size": 3,
            "length_penalty": 0.7,
            "predictions": {
                str(image_id): records[image_id]
                for image_id in TEST_IMAGE_IDS
            },
            "continuation_of_opening_id": OPENING_ID,
            "new_test_opening": False,
            "test_open_count": 1,
            "test_set_used": True,
        },
    )

if str(TEACHER_SOURCE) not in sys.path:
    sys.path.insert(0, str(TEACHER_SOURCE))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

with TEACHER_VOCAB.open("rb") as handle:
    native = pickle.load(handle)
WORD2IDX = native["word2idx_dict"]
IDX2WORD = native["idx2word_list"]
NATIVE_BOS = int(WORD2IDX[native["sos_str"]])
NATIVE_EOS = int(WORD2IDX[native["eos_str"]])
NATIVE_PAD = int(WORD2IDX["PAD"])
NATIVE_UNK = int(WORD2IDX["UNK"])
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (78, 79, 77, 80)

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
teacher = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16, output_word2idx=WORD2IDX,
    output_idx2word=IDX2WORD, max_seq_len=74,
    drop_args=drop_args, rank=device,
)
payload = torch.load(
    TEACHER_CHECKPOINT, map_location="cpu", weights_only=False
)
incompatible = teacher.load_state_dict(payload["model_state_dict"], strict=True)
assert not incompatible.missing_keys and not incompatible.unexpected_keys
del payload
teacher = teacher.to(device).eval()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)

def decode_teacher(values):
    words = []
    for value in values:
        value = int(value)
        if value == NATIVE_BOS:
            continue
        if value == NATIVE_EOS:
            break
        if value not in {NATIVE_PAD, NATIVE_UNK}:
            words.append(str(IDX2WORD[value]))
    return " ".join(words).strip()

@torch.inference_mode()
def generate_teacher():
    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, batch in enumerate(TEACHER_LOADER, start=1):
        ids = [int(value) for value in batch["image_ids"].tolist()]
        images = batch["images"].to(device, non_blocking=True)
        predictions, _ = teacher(
            enc_x=images,
            enc_x_num_pads=[0] * images.shape[0],
            mode="beam_search",
            beam_size=3,
            beam_max_seq_len=31,
            sample_or_max="max",
            how_many_outputs=1,
            sos_idx=NATIVE_BOS,
            eos_idx=NATIVE_EOS,
            length_penalty=0.7,
        )
        for index, image_id in enumerate(ids):
            token_ids = [int(value) for value in predictions[index][0]]
            caption = decode_teacher(token_ids)
            assert caption and image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "image_id": image_id,
                "caption": caption,
                "token_ids": token_ids,
            }
        if batch_index % 100 == 0 or len(hypotheses) == 5000:
            print(
                f"teacher | {len(hypotheses):04d}/5000 | "
                f"{time.perf_counter() - started:.1f}s",
                flush=True,
            )
    return hypotheses, records, time.perf_counter() - started

teacher_hypotheses, teacher_records, teacher_elapsed = generate_teacher()
generated["teacher"] = teacher_hypotheses
generation_seconds["teacher"] = teacher_elapsed
write_json(
    PREDICTIONS / "held_out_test_teacher_predictions.json",
    {
        "schema_version": "held_out_test_comparator_predictions_v1",
        "created_at_utc": now(),
        "model_id": "teacher",
        "seed": None,
        "checkpoint_sha256": sha256(TEACHER_CHECKPOINT),
        "split": "held_out_test",
        "images": len(teacher_records),
        "beam_size": 3,
        "length_penalty": 0.7,
        "predictions": {
            str(image_id): teacher_records[image_id]
            for image_id in TEST_IMAGE_IDS
        },
        "continuation_of_opening_id": OPENING_ID,
        "new_test_opening": False,
        "test_open_count": 1,
        "test_set_used": True,
    },
)

generation_audit = {
    "schema_version": "held_out_test_comparator_generation_v1",
    "created_at_utc": now(),
    "status": "passed",
    "models": [
        "teacher", "baseline_seed42", "baseline_seed123", "baseline_seed2026"
    ],
    "images_per_model": 5000,
    "generation_seconds": generation_seconds,
    "continuation_of_opening_id": OPENING_ID,
    "new_test_opening": False,
    "test_open_count": 1,
    "test_set_used": True,
}
write_json(AUDITS / "comparator_cell03_generation.json", generation_audit)
print("Comparator generation     : PASSED 4/4")

baseline seed=42 | 0800/5000 | 345.0s
baseline seed=42 | 1600/5000 | 617.5s
baseline seed=42 | 2400/5000 | 909.5s
baseline seed=42 | 3200/5000 | 1180.4s
baseline seed=42 | 4000/5000 | 1484.3s
baseline seed=42 | 4800/5000 | 1747.8s
baseline seed=42 | 5000/5000 | 1821.3s
baseline seed=123 | 0800/5000 | 5.1s
baseline seed=123 | 1600/5000 | 9.8s
baseline seed=123 | 2400/5000 | 14.6s
baseline seed=123 | 3200/5000 | 19.4s
baseline seed=123 | 4000/5000 | 24.2s
baseline seed=123 | 4800/5000 | 28.8s
baseline seed=123 | 5000/5000 | 30.0s
baseline seed=2026 | 0800/5000 | 5.2s
baseline seed=2026 | 1600/5000 | 9.8s
baseline seed=2026 | 2400/5000 | 14.4s
baseline seed=2026 | 3200/5000 | 19.0s
baseline seed=2026 | 4000/5000 | 23.5s
baseline seed=2026 | 4800/5000 | 28.1s
baseline seed=2026 | 5000/5000 | 29.2s


/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:206: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  new_loop_dec_classes = loop_dec_classes[[bs_idxes, which_sequence]]
/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:207: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor

teacher | 0400/5000 | 52.9s
teacher | 0800/5000 | 104.9s
teacher | 1200/5000 | 157.2s
teacher | 1600/5000 | 209.4s
teacher | 2000/5000 | 261.3s
teacher | 2400/5000 | 313.4s
teacher | 2800/5000 | 365.5s
teacher | 3200/5000 | 418.0s
teacher | 3600/5000 | 470.5s
teacher | 4000/5000 | 523.1s
teacher | 4400/5000 | 575.5s
teacher | 4800/5000 | 627.7s
teacher | 5000/5000 | 653.9s
Comparator generation     : PASSED 4/4


In [4]:
try:
    from pycocoevalcap.bleu.bleu import Bleu
except ImportError:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl"))
    assert wheels
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(wheels[0])]
    )
    from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

REFERENCES = {
    int(image_id): captions for image_id, captions in PRIOR_REFERENCES.items()
}

def metrics(hypotheses):
    gt = {
        image_id: [{"caption": caption} for caption in captions]
        for image_id, captions in REFERENCES.items()
    }
    pred = {
        image_id: [{"caption": hypotheses[image_id]}]
        for image_id in REFERENCES
    }
    tokenizer = PTBTokenizer()
    gt = tokenizer.tokenize(gt)
    pred = tokenizer.tokenize(pred)
    bleu, _ = Bleu(4).compute_score(gt, pred)
    meteor, _ = Meteor().compute_score(gt, pred)
    rouge, _ = Rouge().compute_score(gt, pred)
    cider, _ = Cider().compute_score(gt, pred)
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }

metric_names = ("BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D")
comparator_metrics = {
    model_id: metrics(hypotheses)
    for model_id, hypotheses in generated.items()
}
kd_metrics = {
    f"kd_seed{seed}": PRIOR_SUMMARY["per_seed_metrics"][str(seed)]
    for seed in (42, 123, 2026)
}
all_metrics = {"teacher": comparator_metrics["teacher"]}
for seed in (42, 123, 2026):
    all_metrics[f"baseline_seed{seed}"] = comparator_metrics[
        f"baseline_seed{seed}"
    ]
    all_metrics[f"kd_seed{seed}"] = kd_metrics[f"kd_seed{seed}"]

family_aggregate = {}
for family in ("baseline", "kd"):
    family_aggregate[family] = {}
    for metric in metric_names:
        values = [
            all_metrics[f"{family}_seed{seed}"][metric]
            for seed in (42, 123, 2026)
        ]
        family_aggregate[family][metric] = {
            "mean": statistics.fmean(values),
            "sample_std": statistics.stdev(values),
        }

paired_delta = {}
for seed in (42, 123, 2026):
    paired_delta[str(seed)] = {
        metric: (
            all_metrics[f"kd_seed{seed}"][metric]
            - all_metrics[f"baseline_seed{seed}"][metric]
        )
        for metric in metric_names
    }

summary = {
    "schema_version": "held_out_test_teacher_baseline_kd_summary_v1",
    "created_at_utc": now(),
    "status": "complete",
    "continuation_of_opening_id": OPENING_ID,
    "models": list(all_metrics),
    "images_per_model": 5000,
    "references": 25010,
    "per_model_metrics": all_metrics,
    "family_mean_and_sample_std": family_aggregate,
    "paired_kd_minus_baseline_by_seed": paired_delta,
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
    "new_test_opening": False,
    "test_open_count": 1,
    "test_set_used": True,
}
summary_path = RESULTS / "HELD_OUT_TEST_TEACHER_BASELINE_KD_SUMMARY.json"
write_json(summary_path, summary)

csv_path = RESULTS / "HELD_OUT_TEST_TEACHER_BASELINE_KD_METRICS.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["model_id", "family", "seed", *metric_names]
    )
    writer.writeheader()
    for model_id, values in all_metrics.items():
        family = (
            "teacher"
            if model_id == "teacher"
            else model_id.split("_seed")[0]
        )
        seed = "" if model_id == "teacher" else model_id.split("seed")[1]
        writer.writerow(
            {
                "model_id": model_id,
                "family": family,
                "seed": seed,
                **values,
            }
        )

report_path = RESULTS / "HELD_OUT_TEST_TEACHER_BASELINE_KD_REPORT.md"
lines = [
    "# Held-out Test — Teacher, Baseline và Dense Mapped Logit-KD",
    "",
    "| Model | Seed | BLEU-1 | BLEU-4 | METEOR | ROUGE-L | CIDEr-D |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for model_id, values in all_metrics.items():
    seed = "—" if model_id == "teacher" else model_id.split("seed")[1]
    lines.append(
        f"| {model_id} | {seed} | {values['BLEU-1']:.6f} | "
        f"{values['BLEU-4']:.6f} | {values['METEOR']:.6f} | "
        f"{values['ROUGE-L']:.6f} | {values['CIDEr-D']:.6f} |"
    )
lines.extend(["", "## Mean ± sample standard deviation", ""])
for family in ("baseline", "kd"):
    lines.append(f"### {family}")
    for metric in metric_names:
        item = family_aggregate[family][metric]
        lines.append(
            f"- {metric}: {item['mean']:.6f} ± {item['sample_std']:.6f}"
        )
lines.extend(
    [
        "",
        "Đây là phần tiếp nối của cùng opening ID; không mở Test lần hai, "
        "không chọn seed/checkpoint và không điều chỉnh phương pháp.",
    ]
)
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

artifacts = [
    OUT / "HELD_OUT_TEST_COMPARATOR_CONTINUATION.json",
    AUDITS / "comparator_cell01_preflight.json",
    AUDITS / "comparator_cell02_data.json",
    AUDITS / "comparator_cell03_generation.json",
    *sorted(PREDICTIONS.glob("*.json")),
    summary_path,
    csv_path,
    report_path,
]
manifest_path = MANIFESTS / "held_out_test_comparator_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "bytes", "sha256"]
    )
    writer.writeheader()
    for path in sorted(artifacts, key=lambda item: str(item)):
        writer.writerow(
            {
                "relative_path": path.relative_to(OUT).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
        )

complete = {
    "schema_version": "held_out_test_comparators_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "continuation_of_opening_id": OPENING_ID,
    "prior_kd_test_marker_sha256": sha256(PRIOR_MARKER_PATH),
    "models_evaluated": list(all_metrics),
    "teacher_count": 1,
    "baseline_seeds": [42, 123, 2026],
    "kd_seeds_reused_from_prior_run": [42, 123, 2026],
    "test_images": 5000,
    "test_references": 25010,
    "summary": {
        "relative_path": summary_path.relative_to(OUT).as_posix(),
        "sha256": sha256(summary_path),
    },
    "metrics_csv": {
        "relative_path": csv_path.relative_to(OUT).as_posix(),
        "sha256": sha256(csv_path),
    },
    "report": {
        "relative_path": report_path.relative_to(OUT).as_posix(),
        "sha256": sha256(report_path),
    },
    "manifest": {
        "relative_path": manifest_path.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_path),
        "rows": len(artifacts),
    },
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
    "new_test_opening": False,
    "test_open_count": 1,
    "test_set_used": True,
}
write_json(COMPLETE_MARKER, complete)

print("Comparator completion     : COMPLETE")
print("Models in final report    :", len(all_metrics))
print("New Test opening          : False")
print("Test open count           : 1")
print("Final marker              :", COMPLETE_MARKER)

Processing /kaggle/input/datasets/nguynthnhdan/dense-mapped-logitkd-held-out-test-static-input/XLKD05_Dense_Mapped_LogitKD_Held_Out_Test_Static_Input/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 307086 tokens at 732685.95 tokens per second.
PTBTokenizer tokenized 53434 tokens at 279325.32 tokens per second.


{'testlen': 48435, 'reflen': 47883, 'guess': [48435, 43435, 38435, 33435], 'correct': [34768, 18389, 9155, 4648]}
ratio: 1.0115280997431029


PTBTokenizer tokenized 307086 tokens at 745647.89 tokens per second.
PTBTokenizer tokenized 53850 tokens at 338199.88 tokens per second.


{'testlen': 48851, 'reflen': 48231, 'guess': [48851, 43851, 38851, 33851], 'correct': [34873, 18468, 9122, 4550]}
ratio: 1.0128548029275568


PTBTokenizer tokenized 307086 tokens at 649479.12 tokens per second.
PTBTokenizer tokenized 53480 tokens at 306511.45 tokens per second.


{'testlen': 48481, 'reflen': 47913, 'guess': [48481, 43481, 38481, 33481], 'correct': [34603, 18369, 9066, 4510]}
ratio: 1.0118548201949156


PTBTokenizer tokenized 307086 tokens at 665248.75 tokens per second.
PTBTokenizer tokenized 52537 tokens at 335695.72 tokens per second.


{'testlen': 47538, 'reflen': 47409, 'guess': [47538, 42538, 37538, 32538], 'correct': [39480, 23917, 12642, 6248]}
ratio: 1.0027210023413065
Comparator completion     : COMPLETE
Models in final report    : 7
New Test opening          : False
Test open count           : 1
Final marker              : /kaggle/working/XLKD05_Dense_Mapped_LogitKD_held_out_test_comparator_completion/XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPARATORS_COMPLETE.json
